# 04a — Week 4 setup & data prep verification

**Goal:** make sure the deep-learning stack on this machine is ready before we
spend an evening fine-tuning BiLSTM / PhoBERT models that fall over halfway
through training because of a missing CUDA toolkit or a 4-dim/300-dim
embedding mismatch.

Parts:

1. **A — Environment verify**: CUDA, GPU, HuggingFace, reproducibility seed.
2. **B — Data loading**: cleaned CSVs from Week 2 (auto-regen if missing).
3. **C — Vocab + embeddings**: build word vocabulary, download PhoW2V, build
   embedding matrix with coverage report.
4. **D — DataLoader smoke test**: one batch through `ViHSDDataset` in both
   `'bilstm'` and `'phobert'` modes.

All artefacts go under `models/dl/`. Random seed is fixed at **42** end-to-end.

In [ ]:
# Auto-reload edited .py files (so editing src/*.py doesn't need a kernel restart).
%load_ext autoreload
%autoreload 2

# ── Common imports & project root ────────────────────────────────────────
import os, sys, time, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from configs.config import PATHS, COLUMNS, LABEL_MAP, DEEP_LEARNING_CONFIG, PHOBERT_CONFIG
from src.utils import set_seed, Timer

RANDOM_STATE = 42
set_seed(RANDOM_STATE)

# Artefact directories for this week.
DL_DIR        = ROOT / "models" / "dl"
EMBED_DIR     = ROOT / "models" / "embeddings"
PROCESSED_DIR = ROOT / "data" / "processed"
for d in (DL_DIR, EMBED_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

TEXT, LABEL = COLUMNS["text"], COLUMNS["label"]
print(f"Project root  : {ROOT}")
print(f"Seed          : {RANDOM_STATE}")
print(f"DL artefacts  : {DL_DIR}")
print(f"Embeddings    : {EMBED_DIR}")

## Part A — Environment verification

In [ ]:
# A1. CUDA & GPU sanity check ───────────────────────────────────────────
import torch

print(f"torch         : {torch.__version__}")
print(f"torch.cuda    : {torch.version.cuda}")
print(f"is_available  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    msg = (
        "\n✗ CUDA is NOT available — STOPPING.\n"
        "Install steps:\n"
        "  1) Verify NVIDIA driver:   nvidia-smi\n"
        "  2) Install CUDA-enabled PyTorch matching your driver, e.g.:\n"
        "       pip install torch --index-url https://download.pytorch.org/whl/cu121\n"
        "     (or cu118 / cu124 depending on driver — see https://pytorch.org)\n"
        "  3) Restart this kernel and re-run this cell.\n"
    )
    raise RuntimeError(msg)

idx = torch.cuda.current_device()
props = torch.cuda.get_device_properties(idx)
vram_gb = props.total_memory / 1024**3
print(f"GPU           : {props.name}")
print(f"VRAM total    : {vram_gb:.2f} GB")
print(f"Compute cap.  : {props.major}.{props.minor}")
print(f"Device index  : {idx} / {torch.cuda.device_count()}")

# RTX 3060 sanity: desktop = 12 GB, mobile = 6 GB. Calibrate batch advice to VRAM.
if vram_gb < 4.0:
    print(f"⚠ Only {vram_gb:.1f} GB VRAM — PhoBERT may not fit; expect frequent OOM.")
elif vram_gb < 8.0:
    print(f"ℹ {vram_gb:.1f} GB VRAM (laptop tier). Recommended PhoBERT settings:")
    print( "    batch_size=8, gradient_accumulation_steps=2  (eff. batch=16)")
    print( "    enable AMP (torch.cuda.amp.autocast / bf16) to halve activations.")
else:
    print(f"✓ {vram_gb:.1f} GB VRAM is comfortable for PhoBERT batch_size=16.")

In [ ]:
# A2. GPU compute smoke test (1000×1000 matmul) ─────────────────────────
import torch, time

# Warm-up (first GPU op pays a one-time context-init cost).
_ = torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")
torch.cuda.synchronize()

x = torch.randn(1000, 1000, device="cuda")
t0 = time.perf_counter()
y = x @ x.T
torch.cuda.synchronize()
dt_ms = (time.perf_counter() - t0) * 1000

print(f"1000×1000 matmul  : {dt_ms:.2f} ms")
print(f"output dtype/dev  : {y.dtype} on {y.device}")
print(f"output norm       : {y.norm().item():.4f}")

if dt_ms >= 100:
    print(f"⚠ Expected < 100 ms on a modern GPU; got {dt_ms:.1f} ms — check thermals / driver.")
else:
    print(f"✓ GPU compute OK ({dt_ms:.1f} ms)")

In [ ]:
# A3. HuggingFace install + PhoBERT tokenizer verify ────────────────────
from transformers import AutoTokenizer

VERIFY_MODEL = "vinai/phobert-base"   # quick lightweight verify; v2 is used in training
print(f"Loading tokenizer: {VERIFY_MODEL} (downloads on first run, ~1-2 MB)")
with Timer("tokenizer-load"):
    tok = AutoTokenizer.from_pretrained(VERIFY_MODEL)

sample = "Hôm nay trời rất đẹp và tôi cảm thấy rất vui."
enc = tok(sample, return_tensors="pt")
tokens = tok.convert_ids_to_tokens(enc["input_ids"][0].tolist())

print(f"  tokenizer type : {type(tok).__name__}")
print(f"  vocab size     : {tok.vocab_size:,}")
print(f"  pad / unk / cls: {tok.pad_token_id} / {tok.unk_token_id} / {tok.cls_token_id}")
print(f"  sample text    : {sample!r}")
print(f"  → {len(tokens)} tokens:")
print(f"    {tokens}")

In [ ]:
# A4. Reproducibility check ─────────────────────────────────────────────
# `set_seed` is the canonical entry point used by every training script;
# here we just confirm two consecutive calls produce identical random draws.
from src.utils import set_seed
import torch, numpy as np, random

def fingerprint() -> dict:
    return {
        "py":    random.random(),
        "np":    float(np.random.rand()),
        "torch": float(torch.rand(1).item()),
        "cuda":  float(torch.rand(1, device="cuda").item()),
    }

set_seed(42); a = fingerprint()
set_seed(42); b = fingerprint()
print("draw 1:", a)
print("draw 2:", b)
assert a == b, "Reproducibility broken — investigate before training."
print("✓ set_seed(42) is reproducible across all RNGs.")

## Part B — Data loading

Re-uses the cleaned splits produced by `02_preprocessing`. The auto-regen
block below mirrors what `03b` does, so this notebook works on a fresh
clone too.

In [ ]:
# B1. Load cleaned splits (auto-regen if missing) ───────────────────────
required_csvs = {split: PROCESSED_DIR / f"{split}_cleaned.csv"
                 for split in ("train", "dev", "test")}
missing = [p for p in required_csvs.values() if not p.exists()]

if missing:
    print(f"⚠ Missing cleaned CSVs: {[p.name for p in missing]}")
    print("→ Regenerating from data/raw/ (~2-3 min)...")
    from src.preprocess import VietnameseTextCleaner, batch_clean
    cleaner = VietnameseTextCleaner()
    for split, out in required_csvs.items():
        df = pd.read_csv(PATHS[f"raw_{split}"])
        df["cleaned"] = batch_clean(df[TEXT], cleaner=cleaner, desc=split)
        df[[TEXT, "cleaned", LABEL]].to_csv(out, index=False)
        print(f"  ✓ {out.name}: {len(df):,} rows")

train_df = pd.read_csv(required_csvs["train"])
dev_df   = pd.read_csv(required_csvs["dev"])
test_df  = pd.read_csv(required_csvs["test"])

# Force string dtype + drop NaN-as-empty
for df in (train_df, dev_df, test_df):
    df["cleaned"] = df["cleaned"].fillna("").astype(str)

print(f"train : {len(train_df):,}  |  label dist: {dict(train_df[LABEL].value_counts().sort_index())}")
print(f"dev   : {len(dev_df):,}    |  label dist: {dict(dev_df[LABEL].value_counts().sort_index())}")
print(f"test  : {len(test_df):,}   |  label dist: {dict(test_df[LABEL].value_counts().sort_index())}")
train_df.head(3)

## Part C — Vocabulary & pretrained embeddings

* Build a word-level vocabulary from `train_cleaned` only (never from
  dev/test — that would leak label-correlated tokens).
* Download a Vietnamese word-embedding file (PhoW2V 100d by default;
  fastText cc.vi.300 is offered as a heavier alternative).
* Build the `(vocab_size, dim)` embedding matrix and report coverage.

In [ ]:
# C1. Build vocab from train_cleaned ────────────────────────────────────
from src.dataset_dl import Vocab

VOCAB_PATH = DL_DIR / "vocab.pkl"
MIN_FREQ   = 2
MAX_SIZE   = 20_000

with Timer("build-vocab"):
    vocab = Vocab.build_from_texts(train_df["cleaned"], min_freq=MIN_FREQ, max_size=MAX_SIZE)

print(f"  vocab size           : {len(vocab):,}  (cap {MAX_SIZE:,}, min_freq {MIN_FREQ})")
print(f"  unique words in train: {len(vocab.freqs):,}")
print(f"  specials             : {vocab.itos[:4]}")

print("\n  top 20 most-common tokens:")
for tok, c in vocab.freqs.most_common(20):
    print(f"    {tok:<20s} {c:>6,}")

vocab.save(VOCAB_PATH)
print(f"\n✓ saved → {VOCAB_PATH}")

In [ ]:
# C2. Choose & download a pretrained embedding file ─────────────────────
# Default: PhoW2V 300d (best for this project — same underthesea tokenisation
# as our cleaned text, same authorship as PhoBERT). The file must be placed
# manually because VinAI's public.vinai.io is offline and the canonical
# source is now Google Drive via the PhoW2V GitHub repo.
#
# To switch source: change EMBEDDING_CHOICE below. The pipeline is
# dim-agnostic — whatever file you point at, build_embedding_matrix will
# detect the dim and produce a matching matrix.
EMBEDDING_CHOICE = "phow2v_300d"

EMBEDDING_OPTIONS = {
    # PRIMARY — manual download required, ~4 GB, 300d
    "phow2v_300d": {
        "url":          None,                                            # manual only
        "path":         EMBED_DIR / "phow2v_300d.txt",
        "dim":          300,
        "download_doc": (
            "Manual download — VinAI's public.vinai.io is offline.\n"
            "  1. Open https://github.com/datquocnguyen/PhoW2V\n"
            "  2. In the README table, find the row 'word' / dim '300'\n"
            "  3. Click the Google Drive link and download "
                  "'word2vec_vi_words_300dims.txt' (~4 GB)\n"
            "  4. If you get a .zip, extract it. The file must be plain text.\n"
            f"  5. Move/rename it to: {EMBED_DIR / 'phow2v_300d.txt'}\n"
            "  6. Re-run this cell — it will skip download and just verify."
        ),
    },
    # Same source, smaller, 100d
    "phow2v_100d": {
        "url":          None,
        "path":         EMBED_DIR / "phow2v_100d.txt",
        "dim":          100,
        "download_doc": (
            "Manual download — see the 100d 'word' row at:\n"
            "  https://github.com/datquocnguyen/PhoW2V\n"
            f"Place the file at: {EMBED_DIR / 'phow2v_100d.txt'}"
        ),
    },
    # AUTO fallback — works without manual steps but lower coverage on our cleaned text
    "fasttext_cc_vi_300d": {
        "url":          "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.vec.gz",
        "path":         EMBED_DIR / "cc.vi.300.vec.gz",
        "dim":          300,
        "download_doc": "Auto-download from dl.fbaipublicfiles.com (~1.5 GB compressed).",
    },
}

choice = EMBEDDING_OPTIONS[EMBEDDING_CHOICE]
print(f"Choice : {EMBEDDING_CHOICE}  (dim={choice['dim']})")
print(f"Target : {choice['path']}")

if choice["path"].exists():
    size_mb = choice["path"].stat().st_size / 1024**2
    print(f"✓ already downloaded ({size_mb:,.1f} MB) — skipping.")
elif choice["url"] is None:
    raise FileNotFoundError(
        f"\n✗ {choice['path']} not found and this choice has no auto-download.\n\n"
        f"{choice['download_doc']}\n"
    )
else:
    import urllib.request
    print(f"⬇ downloading from {choice['url']}")
    print(f"  {choice['download_doc']}")
    t0 = time.perf_counter()

    def _reporthook(blocks, block_size, total_size):
        done = blocks * block_size
        pct  = 100 * done / total_size if total_size > 0 else 0
        if blocks % 200 == 0:
            print(f"    … {done/1024**2:>7.1f} MB ({pct:5.1f}%)")

    urllib.request.urlretrieve(choice["url"], choice["path"], _reporthook)
    dt = time.perf_counter() - t0
    print(f"✓ downloaded {choice['path'].stat().st_size/1024**2:,.1f} MB in {dt:.1f}s")

In [ ]:
# C3. Build embedding matrix ────────────────────────────────────────────
from src.dataset_dl import build_embedding_matrix

EMBEDDING_MATRIX_PATH = DL_DIR / f"embedding_matrix_{EMBEDDING_CHOICE}.pt"

with Timer("build-embedding"):
    emb, stats = build_embedding_matrix(
        vocab             = vocab,
        embedding_path    = choice["path"],
        dim               = choice["dim"],
        unk_init_range    = 0.25,
        seed              = RANDOM_STATE,
        compound_fallback = True,        # average parts of "hôm_nay" if not direct
    )

print(f"  shape           : {tuple(emb.shape)}")
print(f"  dtype           : {emb.dtype}")
print(f"  direct hits     : {stats['found_direct']:,}  ({stats['found_direct']/stats['vocab_size']:.1%})")
print(f"  compound fallback: {stats['found_compound']:,}  (averaged parts)")
print(f"  total coverage  : {stats['found']:,} / {stats['vocab_size']:,}  ({stats['coverage']:.1%})")
print(f"  missing         : {stats['missing']:,}  (random-init in [-0.25, 0.25])")

# Realistic expectations:
#   PhoW2V (same underthesea tokenisation)         → 70-85% direct
#   fastText cc.vi (raw text) + compound fallback  → 50-70% combined
#   fastText cc.vi without fallback                → 25-35% direct
expected_floor = 0.70 if EMBEDDING_CHOICE.startswith("phow2v") else 0.50
if stats["coverage"] < expected_floor:
    print(f"⚠ Coverage {stats['coverage']:.1%} below the {expected_floor:.0%} floor for {EMBEDDING_CHOICE}.")
    print( "  → If using fastText: check that text was cleaned with underthesea (underscore-joined words).")
else:
    print(f"✓ coverage within expected band for {EMBEDDING_CHOICE}.")

torch.save({"embedding": emb, "stats": stats, "choice": EMBEDDING_CHOICE},
           EMBEDDING_MATRIX_PATH)
print(f"\n✓ saved → {EMBEDDING_MATRIX_PATH}")

In [ ]:
# C4. Class weights for the imbalanced 3-class loss ─────────────────────
from src.dataset_dl import get_class_weights

class_weights = get_class_weights(train_df[LABEL])
print("class weights (sklearn 'balanced'):")
for i, w in enumerate(class_weights.tolist()):
    print(f"  {i} = {LABEL_MAP[i]:<10s} → {w:.4f}")

torch.save(class_weights, DL_DIR / "class_weights.pt")
print(f"\n✓ saved → {DL_DIR / 'class_weights.pt'}")

## Part D — `ViHSDDataset` + `DataLoader` smoke test

One batch through each mode. The shapes (and lack of exceptions) are the
deliverable here.

In [ ]:
# D1. BiLSTM mode ───────────────────────────────────────────────────────
from torch.utils.data import DataLoader
from src.dataset_dl import ViHSDDataset, collate_fn_bilstm

BATCH_SIZE = DEEP_LEARNING_CONFIG["batch_size"]
MAX_LEN    = DEEP_LEARNING_CONFIG["max_len"]

train_ds_lstm = ViHSDDataset(
    texts     = train_df["cleaned"].tolist(),
    labels    = train_df[LABEL].tolist(),
    tokenizer = vocab,
    max_len   = MAX_LEN,
    mode      = "bilstm",
)
print(f"len(train_ds_lstm) = {len(train_ds_lstm):,}")
print(f"item[0] keys       = {list(train_ds_lstm[0].keys())}")
print(f"item[0] shapes     = " + str({k: tuple(v.shape) for k, v in train_ds_lstm[0].items()}))

loader = DataLoader(
    train_ds_lstm,
    batch_size = BATCH_SIZE,
    shuffle    = True,
    collate_fn = collate_fn_bilstm,
    generator  = torch.Generator().manual_seed(RANDOM_STATE),
)
batch = next(iter(loader))
print("\nbatch shapes:")
for k, v in batch.items():
    print(f"  {k:<11s} {tuple(v.shape):<14s} {v.dtype}")
print(f"  lengths[:8]  : {batch['lengths'][:8].tolist()}")
print(f"  labels[:8]   : {batch['labels'][:8].tolist()}")

In [ ]:
# D2. PhoBERT mode ──────────────────────────────────────────────────────
from src.dataset_dl import collate_fn_phobert

PHOBERT_NAME = PHOBERT_CONFIG["pretrained_model"]   # vinai/phobert-base-v2 by default
print(f"Loading tokenizer for training: {PHOBERT_NAME}")
with Timer("phobert-tokenizer-load"):
    phobert_tok = AutoTokenizer.from_pretrained(PHOBERT_NAME, use_fast=False)

train_ds_bert = ViHSDDataset(
    texts     = train_df["cleaned"].tolist(),
    labels    = train_df[LABEL].tolist(),
    tokenizer = phobert_tok,
    max_len   = PHOBERT_CONFIG["max_len"],
    mode      = "phobert",
)
print(f"len(train_ds_bert) = {len(train_ds_bert):,}")
print(f"item[0] shapes     = " + str({k: tuple(v.shape) for k, v in train_ds_bert[0].items()}))

loader_bert = DataLoader(
    train_ds_bert,
    batch_size = PHOBERT_CONFIG["batch_size"],
    shuffle    = True,
    collate_fn = collate_fn_phobert,
    generator  = torch.Generator().manual_seed(RANDOM_STATE),
)
batch_bert = next(iter(loader_bert))
print("\nbatch shapes:")
for k, v in batch_bert.items():
    print(f"  {k:<14s} {tuple(v.shape):<14s} {v.dtype}")
print(f"  labels[:8]      : {batch_bert['labels'][:8].tolist()}")

## Summary — what we have, what's next

Saved under `models/`:

* `models/dl/vocab.pkl`                            — word vocabulary (≤ 20 k)
* `models/dl/embedding_matrix_<choice>.pt`         — pretrained embedding matrix
* `models/dl/class_weights.pt`                     — `[w_clean, w_offensive, w_hate]`
* `models/embeddings/<file>`                       — raw download (git-ignored)

**Next (Prompt 2):** `notebooks/04b_bilstm.ipynb` — define and train a
BiLSTM-attention model on top of the saved embedding matrix; target
> baseline ML champion (F1_macro_test = 0.6194).